# Markov Chains, Markov Reward Processes and Bellman Equation

**Reinforcement Learning Laboratory Practical**

## Aim
1. Verify the Markov Property and represent a Markov Chain using a weather-prediction example.
2. Compute state values for a Markov Reward Process (MRP) using the Bellman equation and NumPy.
3. Verify Bellman's recursive equation by comparing manual calculations with Python results.

## Learning outcomes
- Construct and validate a transition-probability matrix.
- Simulate a finite-state Markov Chain.
- Examine the conditional independence expressed by the Markov Property.
- Solve an MRP using both a direct linear-system solution and iterative Bellman updates.
- Verify Bellman's recursive relationship numerically.

> **Data note:** The assignment does not prescribe numerical weather-transition probabilities or state rewards. The transition probabilities and rewards in this notebook are illustrative assumptions for demonstrating the required methods. They are not real weather forecasts.


## Part I — Markov Chain and Markov Property

A **Markov Chain** is a stochastic process in which the next state depends on the current state. The Markov Property is expressed as:

\[
P(X_{t+1}=j\mid X_t=i,X_{t-1},\ldots,X_0)
=
P(X_{t+1}=j\mid X_t=i)
\]

The weather model uses three states: **Sunny**, **Cloudy**, and **Rainy**. The transition matrix entry \(P_{ij}\) gives the probability of moving from current state \(i\) to next state \(j\).


### 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from IPython.display import display

np.random.seed(42)

print("Libraries imported successfully!")

### 2. Define the weather states

In [ ]:
# Define weather states
states = ["Sunny", "Cloudy", "Rainy"]

# Number of states
n_states = len(states)

print("Weather States:", states)
print("Total States:", n_states)

### 3. Construct the transition matrix

The following probabilities are illustrative. Each row represents the current weather, and each column represents the next day's weather.


In [ ]:
# Define transition matrix
P = np.array([
    [0.7, 0.2, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])

transition_df = pd.DataFrame(
    P,
    index=states,
    columns=states
)

print("Weather Transition Matrix:")
display(transition_df)

### 4. Validate the transition matrix

A valid transition matrix must have non-negative entries and each row must sum to 1.


In [ ]:
row_sums = P.sum(axis=1)

print("Sum of each row:")

for state, total in zip(states, row_sums):
    print(f"{state}: {total:.2f}")

if np.allclose(row_sums, 1):
    print("\nValid Transition Matrix!")
else:
    print("\nInvalid Transition Matrix!")

### 5. Draw the weather Markov Chain

In [ ]:
G = nx.DiGraph()

for state in states:
    G.add_node(state)

for i in range(n_states):
    for j in range(n_states):

        probability = P[i, j]

        if probability > 0:
            G.add_edge(
                states[i],
                states[j],
                weight=probability
            )

pos = {
    "Sunny": (0, 1),
    "Cloudy": (2, 1),
    "Rainy": (1, -1)
}

plt.figure(figsize=(9, 6))

nx.draw_networkx_nodes(
    G, pos,
    node_size=2500,
    node_color="lightblue"
)

nx.draw_networkx_labels(
    G, pos,
    font_size=12,
    font_weight="bold"
)

nx.draw_networkx_edges(
    G, pos,
    arrows=True,
    arrowsize=20,
    connectionstyle="arc3,rad=0.12"
)

edge_labels = nx.get_edge_attributes(G, "weight")

edge_labels = {
    edge: f"{weight:.1f}"
    for edge, weight in edge_labels.items()
}

nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels=edge_labels,
    font_size=10
)

plt.title("Weather Prediction Markov Chain")
plt.axis("off")
plt.show()

### 6. Simulate weather over 30 days

The simulation starts in the Sunny state. At each step, the next state is sampled using the transition probabilities associated with the current state.


In [ ]:
def simulate_weather(initial_state, days):

    current_state = states.index(initial_state)

    weather_sequence = [initial_state]

    for day in range(days - 1):

        next_state = np.random.choice(
            n_states,
            p=P[current_state]
        )

        weather_sequence.append(states[next_state])

        current_state = next_state

    return weather_sequence


weather_sequence = simulate_weather("Sunny", 30)

weather_df = pd.DataFrame({
    "Day": range(1, 31),
    "Weather": weather_sequence
})

display(weather_df)

### 7. Visualize the simulated weather sequence

In [ ]:
weather_numeric = [
    states.index(weather)
    for weather in weather_sequence
]

plt.figure(figsize=(12, 5))

plt.step(
    range(1, 31),
    weather_numeric,
    where="mid",
    marker="o"
)

plt.yticks(
    range(3),
    states
)

plt.xlabel("Day")
plt.ylabel("Weather Condition")

plt.title("Weather Prediction Using Markov Chain")

plt.grid(alpha=0.3)
plt.show()

### 8. Empirical examination of the Markov Property

This section compares the empirical distribution of **all three possible next states** for every observed previous-state/current-state pair against the transition-matrix row for the current state.

For a fixed current state, the theoretical next-state distribution must be the same regardless of the previous state. Because this is a finite simulation, empirical probabilities will not match the theoretical values exactly; small sampling differences are expected.


In [ ]:
# Generate a long sequence to estimate transition probabilities empirically
large_sequence = simulate_weather("Sunny", 200000)

# Convert weather names into numerical state indices
sequence = np.array([states.index(weather) for weather in large_sequence])

results = []
max_absolute_error = 0.0

# Compare every possible previous-state/current-state pair.
# Under the Markov Property, once the current state is known,
# the distribution of the next state should not depend on the previous state.
for previous_state in range(n_states):
    for current_state in range(n_states):

        pair_mask = (
            (sequence[:-2] == previous_state) &
            (sequence[1:-1] == current_state)
        )

        observed_next_states = sequence[2:][pair_mask]
        observation_count = len(observed_next_states)

        if observation_count == 0:
            continue

        empirical_probabilities = np.bincount(
            observed_next_states,
            minlength=n_states
        ) / observation_count

        theoretical_probabilities = P[current_state]

        for next_state in range(n_states):
            empirical_probability = empirical_probabilities[next_state]
            theoretical_probability = theoretical_probabilities[next_state]

            absolute_error = abs(
                empirical_probability - theoretical_probability
            )

            max_absolute_error = max(max_absolute_error, absolute_error)

            results.append({
                "Previous State": states[previous_state],
                "Current State": states[current_state],
                "Next State": states[next_state],
                "Empirical Probability": empirical_probability,
                "Theoretical Probability": theoretical_probability,
                "Absolute Error": absolute_error,
                "Observations": observation_count
            })

markov_df = pd.DataFrame(results)

print("Empirical vs Theoretical Next-State Probabilities")
display(markov_df.round(4))

print(f"Maximum absolute probability difference: {max_absolute_error:.4f}")
print(
    "Interpretation: For each fixed current state, the estimated next-state "
    "distribution is compared across different previous states. The results "
    "should be close to the same transition-matrix row, with small sampling "
    "differences because the sequence is finite."
)


## Part II — Markov Reward Process (MRP)

A **Markov Reward Process** consists of:
- a state space,
- a transition-probability matrix,
- a reward function, and
- a discount factor.

For this illustrative example, rewards are assigned to weather states: Sunny = 5, Cloudy = 2, and Rainy = −2. The discount factor is \(\gamma=0.9\).

The state-value Bellman equation is:

\[
V(s)=R(s)+\gamma\sum_{s'}P(s'\mid s)V(s')
\]

In matrix notation:

\[
V=R+\gamma PV
\]


### 9. Define the reward function and discount factor

In [ ]:
# Define rewards for each weather state
R = np.array([5, 2, -2])

# Discount factor
gamma = 0.9

reward_df = pd.DataFrame({
    "Weather State": states,
    "Reward": R
})

display(reward_df)

print("Discount Factor:", gamma)

### 10. Solve the Bellman equation using NumPy

Rearranging \(V=R+\gamma PV\):

\[
(I-\gamma P)V=R
\]

Therefore:

\[
V=(I-\gamma P)^{-1}R
\]

The implementation uses `numpy.linalg.solve`, which solves the linear system directly without explicitly calculating a matrix inverse.


In [ ]:
# Identity matrix
I = np.eye(n_states)

# Construct Bellman equation matrix
A = I - gamma * P

# Solve using NumPy
V = np.linalg.solve(A, R)

print("State Values:")

for state, value in zip(states, V):
    print(f"V({state}) = {value:.4f}")

### 11. Display the MRP state values

In [ ]:
mrp_results = pd.DataFrame({
    "Weather State": states,
    "Immediate Reward": R,
    "State Value": V
})

display(mrp_results)

### 12. Verify the solution using iterative Bellman updates

Starting from zero state values, repeatedly apply:

\[
V_{k+1}=R+\gamma PV_k
\]

The process stops when the maximum absolute change between consecutive value vectors is smaller than the tolerance \(\theta\).


In [ ]:
# Initialize values
V_iterative = np.zeros(n_states)

theta = 0.000001

iteration = 0

history = []

while True:

    new_V = R + gamma * np.dot(P, V_iterative)

    delta = np.max(
        np.abs(new_V - V_iterative)
    )

    V_iterative = new_V.copy()

    history.append(V_iterative.copy())

    iteration += 1

    if delta < theta:
        break


print("Converged after", iteration, "iterations")

for state, value in zip(states, V_iterative):
    print(f"V({state}) = {value:.4f}")

### 13. Visualize the convergence of state values

In [ ]:
history_array = np.array(history)

plt.figure(figsize=(10, 5))

for i, state in enumerate(states):

    plt.plot(
        history_array[:, i],
        label=state
    )

plt.xlabel("Iteration")
plt.ylabel("State Value")

plt.title("Convergence of Markov Reward Process")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

## Part III — Verification of Bellman's Recursive Equation

For each state, Bellman's equation expresses its value as its immediate reward plus the discounted expected value of the next state.

For example, for Sunny:

\[
V( Sunny )=5+0.9[0.7V(Sunny)+0.2V(Cloudy)+0.1V(Rainy)]
\]

The next cells calculate the recursive values manually and with NumPy, then compare the results.


### 14. Calculate the Bellman values manually

In [ ]:
# Manual Bellman calculation for Sunny
V_sunny_manual = 5 + 0.9 * (
    0.7 * V[0] +
    0.2 * V[1] +
    0.1 * V[2]
)

# Manual Bellman calculation for Cloudy
V_cloudy_manual = 2 + 0.9 * (
    0.3 * V[0] +
    0.4 * V[1] +
    0.3 * V[2]
)

# Manual Bellman calculation for Rainy
V_rainy_manual = -2 + 0.9 * (
    0.2 * V[0] +
    0.3 * V[1] +
    0.5 * V[2]
)

manual_values = np.array([
    V_sunny_manual,
    V_cloudy_manual,
    V_rainy_manual
])

print("Manually Calculated Values")

for state, value in zip(states, manual_values):
    print(f"V({state}) = {value:.4f}")

### 15. Calculate the Bellman values using Python and NumPy

In [ ]:
python_values = R + gamma * np.dot(P, V)

print("Python Calculated Values")

for state, value in zip(states, python_values):
    print(f"V({state}) = {value:.4f}")

### 16. Compare manual and Python calculations

In [ ]:
comparison_df = pd.DataFrame({

    "Weather State": states,

    "Manual Calculation": manual_values,

    "Python Calculation": python_values,

    "Absolute Difference": np.abs(
        manual_values - python_values
    )

})

display(comparison_df)

if np.allclose(manual_values, python_values):
    print("Verification Successful!")
    print("Bellman's Recursive Equation is Verified.")
else:
    print("Verification Failed.")

### 17. Cross-check direct and iterative MRP solutions

In [ ]:
# Compare direct matrix solution and iterative solution

verification = pd.DataFrame({

    "Weather State": states,

    "Matrix Solution": V,

    "Iterative Solution": V_iterative,

    "Difference": np.abs(V - V_iterative)

})

display(verification)

if np.allclose(V, V_iterative, atol=1e-5):
    print("All MRP calculations are consistent!")
else:
    print("Check the calculations.")

## Observations and Conclusion

Complete this section after executing all cells.

### Observations
1. The transition matrix is valid when every probability is non-negative and each row sums to one.
2. The simulated weather sequence demonstrates state transitions according to the selected transition probabilities.
3. For a Markov Chain, the next-state distribution is determined by the current state. Empirical estimates from a finite simulation may differ slightly from theoretical probabilities because of sampling variation.
4. The MRP state values represent expected discounted cumulative rewards from each starting weather state.
5. The direct NumPy solution and iterative Bellman updates should converge to the same state values within numerical tolerance.
6. Manual Bellman calculations and Python calculations should agree, verifying the recursive equation.

### Conclusion
This practical implements a weather-based Markov Chain, examines the Markov Property, and constructs a Markov Reward Process. The Bellman equation is solved using both a direct linear-system method and iterative updates. Finally, manual calculations are compared with Python results to verify Bellman's recursive relationship.

### Reproducibility note
The random seed is fixed in the import cell to make the simulation reproducible in a fresh execution. Small empirical probability differences are normal and depend on the simulated sample.
